# Using the Qwen model

Build it, start it, ask it things, close it. Settings come from
`configs/qwen.toml`; pass an argument to `QwenModel(...)` to override one.

Each cell is one call.

In [ ]:
from engine.models.base_model import Prompt
from engine.models.qwen_model import QwenModel

## Is the GPU free?

The model needs ~21 GB. Anything listed here is holding the card -- usually a
previous run that was never killed.

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_gpu_memory --format=csv

## Build

Free: nothing loads yet. There is only ever one model: calling `QwenModel()`
again returns this same one. Passing a different setting later restarts the
server with it -- refused while prompts are running.

In [21]:
model = QwenModel()
{k: v for k, v in vars(model).items() if not k.startswith("_")}

{'model_path': '/var/tmp/ge74wan/hf_cache/hub/models--unsloth--Qwen3.8-27B-GGUF/blobs/3f227079003add2511437e5b1e94812e363385225bf6a9b47b0054a72bc8b01e',
 'n_ctx': 8192,
 'seed': 0,
 'temperature': 0.0,
 'top_p': 1.0,
 'top_k': 0,
 'max_tokens': 2000,
 'enable_thinking': False,
 'min_vram_mib': 18000,
 'verbose': False}

## Start

Checks the GPU, starts `llama-server` with 4 slots (~15 s once the weights are
cached, ~40 s cold), then asks the model to say hello and refuses if the
answer isn't language. Safe to run twice: the second call does nothing.

In [22]:
%time model.ready()

CPU times: user 2.26 s, sys: 1 s, total: 3.26 s
Wall time: 3.46 s


## Use

`generate` takes a `Prompt` and returns a `Generation`: the raw text plus what
it cost.

In [23]:
g = model.generate(
    Prompt(
        system="You are a concise assistant.",
        user="write a short poem about the sun and the moon.",
    )
)
print(g.text)

The sun burns bright in golden day,
Chasing shadows far away.
The moon rises in silver light,
To hold the stars within her sight.

One ends where the other begins,
A dance of dusk and dawn’s thin lines.
Together they keep the world in tune,
The sun and the moon, the day and the moon.


In [9]:
g.usage, g.latency_ms, g.stop_reason, round(g.usage.completion_tokens / (g.latency_ms / 1000), 1)

(Usage(prompt_tokens=51, completion_tokens=848), 19408, 'stop', 43.7)

### Constrained output

Pass a JSON schema and a grammar forces the answer into that shape. It costs
~4x decode speed, so leave `schema` off when free text will do.

In [18]:
schema = {
    "type": "object",
    "properties": {
        "ingredient": {"type": "string"},
        "purposes": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["ingredient", "purposes"],
}

g = model.generate(
    Prompt(
        system="Answer in JSON.",
        user="Name one common supplement and what it is taken for, non Vitamins!",
        schema=schema,
    )
)
print(g.text)

{ "ingredient": "Magnesium", "purposes": ["muscle relaxation", "improving sleep quality", "supporting nerve function"] }


## Many at once

The controller hands the model the whole set: the model plans it (measures
every prompt, sets aside any that can't fit, primes its slots), and the
controller keeps all 4 slots busy, retrying failures and reporting progress.

In [ ]:
from engine.controllers.qwen_controller import QwenController

prompts = [
    Prompt(system="You are a concise assistant.", user=f"In one sentence, what is {name} usually taken for?")
    for name in ["magnesium glycinate", "melatonin", "ashwagandha", "vitamin D3", "fish oil", "zinc", "turmeric", "iron"]
]
outcomes = QwenController(model).handle(prompts, on_progress=lambda p: print(p.done, "/", p.total, p.detail))
[o.generation.text for o in outcomes]

## Close

Stops the server and frees the GPU. The server also dies with this kernel, so
a restart never leaves it holding the card.

In [24]:
model.close()